# <font color="steelblue">18. Series cronológicas: componentes y lectura de gráficos temporales</font>

*Unidad 3. Números índice y series cronológicas* · Sesión teórica 18 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`paro_mensual.csv`, `visitas_diario.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["paro_mensual.csv", "visitas_diario.csv"]
PREGUNTAS = ["q_P18_1.json", "q_P18_2.json", "q_P18_3.json", "q_P18_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«El paro baja en julio» es casi siempre verdad... y casi nunca es noticia: pasa todos los años por la temporada turística. Una **serie cronológica** (los datos de una variable ordenados en el tiempo) mezcla una **tendencia**, un patrón **estacional** que se repite y un componente **irregular**. Distinguirlos es imprescindible para no confundir lo habitual con lo excepcional.

**Qué necesitas saber antes**: el cuaderno 14 (tasas interanuales).

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Representar una serie cronológica con un gráfico de líneas.
2. Identificar tendencia, estacionalidad, ciclo e irregularidad.
3. Agregar datos diarios a semanas o meses.
4. Comparar cada periodo con el mismo periodo del año anterior.

## <font color='steelblue'>Contenidos</font>

1. El gráfico de líneas
2. Componentes de una serie
3. Comparar con el mismo periodo del año anterior
4. De datos diarios a semanales o mensuales

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. El gráfico de líneas</font>

In [ ]:
paro = pd.read_csv("paro_mensual.csv")
paro["fecha"] = pd.to_datetime(paro["fecha"])
paro = paro.set_index("fecha")
paro.head()

En una serie el eje horizontal es **el tiempo** y los puntos se unen con líneas. A diferencia de las barras, aquí el eje vertical **no tiene por qué empezar en cero**, siempre que se vea la escala.

In [ ]:
paro["parados"].plot(color="steelblue", figsize=(11, 4))
plt.title("Paro registrado mensual (datos simulados)")
plt.ylabel("Personas")
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P18_1.json")

## <font color='steelblue'>2. Componentes de una serie</font>

Una serie se piensa como la combinación de:

| Componente | Qué es | En el paro |
| :-- | :-- | :-- |
| **Tendencia** | Evolución a largo plazo | Baja durante años |
| **Ciclo** | Oscilaciones de varios años | Crisis y recuperaciones |
| **Estacionalidad** | Patrón que se repite cada año (o semana) | Baja en verano, sube en otoño |
| **Irregular** | Lo que queda: ruido y sucesos puntuales | Una huelga, un temporal |

El salto de 2020 es un componente irregular muy grande (o un cambio de ciclo).

**Qué vamos a hacer**: ver la estacionalidad poniendo cada año en una línea, con los meses en el eje horizontal.

In [ ]:
tabla = paro.assign(anio=paro.index.year, mes=paro.index.month).pivot(index="mes", columns="anio", values="parados")
tabla.plot(figsize=(11, 4), colormap="viridis", legend=False)
plt.xticks(range(1, 13), ["E", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
plt.title("Cada línea es un año: el patrón estacional se repite")
plt.ylabel("Parados")
plt.show()

**Cómo leerlo**: todos los años el paro baja de primavera a verano y sube en otoño. Ese movimiento **no es noticia**; lo que lo es, es que un julio baje más o menos de lo habitual.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P18_2.json")

## <font color='steelblue'>3. Comparar con el mismo periodo del año anterior</font>

Para quitar la estacionalidad sin fórmulas, se compara cada mes con el **mismo mes del año anterior** (tasa interanual):

In [ ]:
paro["var_mensual"] = paro["parados"].diff()
paro["var_interanual"] = paro["parados"].diff(12)
paro.loc["2025-05-01":"2025-09-01", ["parados", "var_mensual", "var_interanual"]]

**Cómo leerlo**: el paro de julio baja respecto a junio (efecto verano), pero lo relevante es si baja respecto a julio del año anterior. Así se escribe: «el paro baja en julio en X personas, la mayor caída interanual desde...».

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P18_3.json")

## <font color='steelblue'>4. De datos diarios a semanales o mensuales</font>

In [ ]:
visitas = pd.read_csv("visitas_diario.csv")
visitas["fecha"] = pd.to_datetime(visitas["fecha"])
visitas = visitas.set_index("fecha")["visitas"]

Los datos diarios tienen mucho ruido y una estacionalidad **semanal** (los fines de semana se lee menos). `resample()` agrega por semana (`"W"`) o por mes (`"MS"`).

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
visitas.plot(ax=axes[0], lw=0.6, title="Diario")
visitas.resample("W").sum().plot(ax=axes[1], title="Semanal (suma)")
visitas.resample("MS").sum().plot(ax=axes[2], marker="o", title="Mensual (suma)")
plt.tight_layout()
plt.show()

In [ ]:
dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
media_dia = visitas.groupby(visitas.index.dayofweek).mean()
media_dia.index = dias
media_dia.round(0)

**Cómo leerlo**: en la serie diaria se ven picos informativos (grandes noticias) y la caída de agosto; la serie mensual muestra mejor la tendencia creciente. El sábado es el día con menos visitas.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P18_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Dibuja la serie de paro solo desde 2022.

<details><summary>Ver solución</summary>

```python
paro.loc["2022":, "parados"].plot()
plt.show()
```
</details>

### **Ejercicio 2** *(sección 2)*

¿En qué mes del año suele haber menos parados? Calcula la media de cada mes del año a lo largo de toda la serie.

<details><summary>Ver solución</summary>

```python
paro.groupby(paro.index.month)["parados"].mean().round()
```
</details>

### **Ejercicio 3** *(sección 3)*

¿Cuál fue la mayor caída interanual del paro y en qué mes?

<details><summary>Ver solución</summary>

```python
paro["var_interanual"].idxmin(), paro["var_interanual"].min()
```
</details>

### **Ejercicio 4** *(sección 4)*

Busca los cinco días con más visitas del diario. ¿Qué tienen en común?

<details><summary>Ver solución</summary>

```python
visitas.sort_values(ascending=False).head()
```

Son picos informativos: en los datos reales coincidirían con grandes noticias.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- Una **serie cronológica** se representa con **líneas**; el eje vertical puede no empezar en cero si la escala está clara.
- Componentes: **tendencia**, **ciclo**, **estacionalidad** e **irregular**.
- Lo estacional se repite cada año y **no es noticia**; compara con el **mismo periodo del año anterior** (`diff(12)`, `pct_change(12)`).
- `resample()` agrega datos diarios a semanas o meses y reduce el ruido.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Serie cronológica** | Valores de una variable ordenados en el tiempo. |
| **Tendencia** | Evolución a largo plazo. |
| **Estacionalidad** | Patrón que se repite con periodo fijo. |
| **Componente irregular** | Variación no sistemática. |
| **Variación interanual** | Diferencia con el mismo periodo del año anterior. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `resample` da `TypeError` | El índice no es de tipo fecha | Usa `pd.to_datetime` y `set_index` |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 19 aislarás la tendencia con medias móviles y medirás la estacionalidad.

**Siguiente cuaderno**: [19. Tendencia con medias móviles y desestacionalización](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P19_tendencia_y_estacionalidad.ipynb)